In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import f1_score, classification_report
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

CSV_PATH = Path("data/cachacaNER.csv")  # ajuste se necessário
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

df = pd.read_csv(CSV_PATH)

In [3]:
for cand in ["sentence_id", "sentence", "sent_id"]:
    if cand in df.columns:
        SENT_COL = cand
        break
else:
    raise KeyError(f"Coluna de sentença não encontrada em {list(df.columns)}")


def sent_to_record(sent_id, g):
    return {
        "sentence_id": int(sent_id),
        "tokens": g["token"].tolist(),
        "ner_tags": g["tag"].tolist(),
    }


In [4]:
records = [sent_to_record(i, g) for i, g in df.groupby(SENT_COL, sort=False)]
cachaca_full = Dataset.from_list(records)

In [5]:
cachaca_full

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 13628
})

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({lab for sent in cachaca_full["ner_tags"] for lab in sent})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-CARACTERISTICA_SENSORIAL_AROMA',
 1: 'B-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 2: 'B-CARACTERISTICA_SENSORIAL_COR',
 3: 'B-CARACTERISTICA_SENSORIAL_SABOR',
 4: 'B-CLASSIFICACAO_BEBIDA',
 5: 'B-EQUIPAMENTO_DESTILACAO',
 6: 'B-GRADUACAO_ALCOOLICA',
 7: 'B-NOME_BEBIDA',
 8: 'B-NOME_LOCAL',
 9: 'B-NOME_ORGANIZACAO',
 10: 'B-NOME_PESSOA',
 11: 'B-PRECO',
 12: 'B-RECIPIENTE_ARMAZENAMENTO',
 13: 'B-TEMPO',
 14: 'B-TEMPO_ARMAZENAMENTO',
 15: 'B-TIPO_MADEIRA',
 16: 'B-VOLUME',
 17: 'I-CARACTERISTICA_SENSORIAL_AROMA',
 18: 'I-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 19: 'I-CARACTERISTICA_SENSORIAL_COR',
 20: 'I-CARACTERISTICA_SENSORIAL_SABOR',
 21: 'I-CLASSIFICACAO_BEBIDA',
 22: 'I-EQUIPAMENTO_DESTILACAO',
 23: 'I-GRADUACAO_ALCOOLICA',
 24: 'I-NOME_BEBIDA',
 25: 'I-NOME_LOCAL',
 26: 'I-NOME_ORGANIZACAO',
 27: 'I-NOME_PESSOA',
 28: 'I-PRECO',
 29: 'I-RECIPIENTE_ARMAZENAMENTO',
 30: 'I-TEMPO',
 31: 'I-TEMPO_ARMAZENAMENTO',
 32: 'I-TIPO_MADEIRA',
 33: 'I-VOLUME',
 34: 'O'}

In [8]:
NUM_LABELS

35

# Splits

In [9]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [10]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [11]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [12]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [13]:
def loc_split(
    dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
) -> DatasetDict:
    """
    Split baseado em baixa sobreposição léxica (4-gram Jaccard).
    Teste = pct_test das sentenças com menor overlap em relação ao pool.
    """
    # 1. Texto plano por sentença
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    # 2. Vetorizar 4-grams (binário)
    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)

    # 3. Similaridade Jaccard aproximada com matriz binária
    # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
    # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
    bin_counts = X.sum(axis=1).A1

    # Para cada doc i, escolhemos vizinho + próximo (fast):
    from sklearn.metrics.pairwise import cosine_similarity

    # (cosine no binário ∝ |A∩B|)
    sim = cosine_similarity(X, dense_output=False)
    # Soma dos top-k overlaps (k=5) como score
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        # overlap ≈ |∩|
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
    order = np.argsort(topk)
    n_test = int(len(dataset) * pct_test)
    test_idx = order[:n_test]
    train_idx = order[n_test:]

    return DatasetDict(
        {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
    )

In [14]:
# standard_split = split_standard(cachaca_full)
# print('std')
# random_splt = random_splits(cachaca_full)
# print('random')
# heur_len = split_heur_length(cachaca_full)
# print("heur_len")
# heur_rare = split_heur_rare(cachaca_full)
# print("heur_rare")
# advers = split_adversarial_fast(cachaca_full)
# print("advs")
print("loc")
loc = loc_split(cachaca_full)

loc


/tmp/ipykernel_73959/216780701.py:36: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()


# Tokenização e Métricas

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# def tokenize(batch):
#     enc = tokenizer(
#         batch["tokens"], is_split_into_words=True, truncation=True, padding=False
#     )
#     new_labels = []
#     for i, word_ids in enumerate(enc.word_ids(batch_index=None)):
#         sent_labels = batch["ner_tags"][i]
#         aligned = []
#         last = None
#         for wid in word_ids:
#             if wid is None:
#                 aligned.append(-100)
#             elif wid != last:
#                 aligned.append(label2id[sent_labels[wid]])
#                 last = wid
#             else:
#                 aligned.append(-100)
#         new_labels.append(aligned)
#     enc["labels"] = new_labels
#     return enc

In [16]:
data_collator = DataCollatorForTokenClassification(
    tokenizer,
    padding=True,  # deixa o collator completar se faltar
)

In [17]:
def tokenize(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding=True,  # garante mesmo comprimento já aqui
        return_attention_mask=True
    )

    all_labels = []
    for i in range(len(batch["tokens"])):  # cada frase
        word_ids = enc.word_ids(batch_index=i)  # INDEX
        sent_tags = batch["ner_tags"][i]

        aligned = []
        last_wid = None
        for wid in word_ids:
            if wid is None: 
                aligned.append(-100)
            elif wid != last_wid:
                aligned.append(label2id[sent_tags[wid]])
                last_wid = wid
            else:  
                aligned.append(-100)

        all_labels.append(aligned)

    enc["labels"] = all_labels
    return enc

In [18]:
def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)
    true_preds, true_labels = [], []
    for p_i, l_i in zip(preds, labels):
        mask = l_i != -100
        true_preds.append([id2label[idx] for idx in p_i[mask]])
        true_labels.append([id2label[idx] for idx in l_i[mask]])
    return {"f1": f1_score(true_labels, true_preds)}


# def run_experiment(dsdict: DatasetDict, seed: int) -> float:
#     encoded = dsdict.map(
#         tokenize, batched=True, remove_columns=dsdict["train"].column_names
#     )
#     model = AutoModelForTokenClassification.from_pretrained(
#         MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
#     )
#     args = TrainingArguments(
#         output_dir=f"tmp/seed{seed}",
#         eval_strategy="no",
#         logging_strategy="steps",
#         logging_steps=50,  # → ≥ 1
#         learning_rate=2e-5,
#         per_device_train_batch_size=16,
#         num_train_epochs=3,
#         seed=seed,
#         save_strategy="no",
#         report_to="none",
#     )
#     trainer = Trainer(
#         model=model,
#         args=args,
#         data_collator=data_collator,
#         train_dataset=encoded["train"],
#         compute_metrics=compute_metrics,
#     )
#     trainer.train()
#     return trainer.evaluate(encoded["dev"])["eval_f1"]


def run_experiment(dsdict: DatasetDict, seed: int) -> float:
    encoded = dsdict.map(
        tokenize, batched=True, remove_columns=dsdict["train"].column_names
    )

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
        torch_dtype="auto",  # usa fp16 se possível
    )

    args = TrainingArguments(
        output_dir=f"tmp/seed{seed}",
        eval_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=2,  # <<< menor
        gradient_accumulation_steps=8,  # <<< compensa
        num_train_epochs=3,
        fp16=True,  # tente first; se falhar, tire
        seed=seed,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=data_collator,  # já sugerido p/ padding labels
        train_dataset=encoded["train"],
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer, encoded

# Experimentos

In [19]:
# print("• Standard split")
# std_f1 = run_experiment(standard_split, seed=SEED_GLOBAL)

In [20]:
# print("• 30 random splits")
# rand_f1 = run_experiment(random_splt, seed=SEED_GLOBAL)

In [21]:
# print("• Heuristic length")
# f1_len = run_experiment(heur_len, seed=SEED_GLOBAL)

In [22]:
# print("• Heuristic rare")
# f1_rare = run_experiment(heur_rare, seed=SEED_GLOBAL)

In [23]:

# print("• Adversarial")
# f1_adv = run_experiment(advers, seed=SEED_GLOBAL)

In [24]:
print("• LOC")
trainer, encoded = run_experiment(loc, seed=SEED_GLOBAL)

• LOC


Map: 100%|██████████| 2725/2725 [00:00<00:00, 7292.62 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,1.460600
100,0.516500
150,0.337700
200,0.239300
250,0.191700
300,0.185300
350,0.159000
400,0.133200
450,0.113100
500,0.118100


In [25]:
f1_metrics_loc = trainer.evaluate(encoded["test"])["eval_f1"]

In [26]:
print("\n=====  RESULTADOS  =====")
# print(f"Standard F1 : {std_f1:.3f}")
# print(f"Random   F1 : {rand_f1:.3f}")
# print(f"Heur-len F1 : {f1_len:.3f}")
# print(f"Heur-rare F1: {f1_rare:.3f}")
# print(f"Advers.  F1 : {f1_adv:.3f}")
print(f"LOC.  F1 : {f1_metrics_loc:.3f}")


=====  RESULTADOS  =====
LOC.  F1 : 0.834


In [27]:
# from transformers import TrainingArguments
# import inspect, os, pathlib

# print("TrainingArguments vindo de:", inspect.getfile(TrainingArguments))
# print(
#     "Tem avaliação?:",
#     "evaluation_strategy" in inspect.signature(TrainingArguments).parameters,
# )